In [ ]:
# ============================================================
# Washing Machine — CNN-BiLSTM Seq2Point v3
# NILM Project | KASIT - University of Jordan
# Supervised by Dr. Ruba Obiedat
# ============================================================
# Key changes from v2:
#   - Val sample reflects real distribution (2.4% ON)
#     instead of 50/50 — model learns precision not memorization
#   - Focal Loss instead of BCE — focuses on hard examples
#   - pos_weight=10 — compensates imbalance without collapse
# ============================================================


# ============================================================
# CELL 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (f1_score, precision_score,
                             recall_score, confusion_matrix)

BASE_PATH    = "/content/drive/MyDrive/new obada/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODELS_PATH  = BASE_PATH + "models/"
os.makedirs(MODELS_PATH, exist_ok=True)

WINDOW_SIZE  = 599
BATCH_SIZE   = 512
EPOCHS       = 40
PATIENCE     = 8
RANDOM_SEED  = 42

# Val sample: reflects real distribution
# 15,069 ON + 6x OFF = ~90,414 OFF → ratio ≈ 6:1
N_VAL_ON     = 15_069
VAL_OFF_MULT = 6   # OFF multiplier relative to ON in val sample

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"✅ Config ready | device={device}")


# ============================================================
# CELL 3: Dataset
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y_label):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y_label, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ============================================================
# CELL 4: Focal Loss + Model
# ============================================================
class FocalLoss(nn.Module):
    """
# Focal Loss: focuses on hard examples, ignores easy ones
# alpha: weight of the ON class
  # gamma: higher value = more focus on misclassified examples
  """
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce  = nn.functional.binary_cross_entropy_with_logits(
                   logits, targets, reduction='none')
        prob = torch.sigmoid(logits)
        p_t  = prob * targets + (1 - prob) * (1 - targets)
        a_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = a_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()


class CNNBiLSTM(nn.Module):
    def __init__(self, window=599):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(1,  30, kernel_size=10, padding=4),
            nn.BatchNorm1d(30), nn.ReLU(),
            nn.Conv1d(30, 30, kernel_size=8,  padding=3),
            nn.BatchNorm1d(30), nn.ReLU(),
            nn.Conv1d(30, 40, kernel_size=6,  padding=2),
            nn.BatchNorm1d(40), nn.ReLU(),
            nn.Conv1d(40, 50, kernel_size=5,  padding=2),
            nn.BatchNorm1d(50), nn.ReLU(),
            nn.Conv1d(50, 50, kernel_size=5,  padding=2),
            nn.BatchNorm1d(50), nn.ReLU(),
        )

        self.lstm = nn.LSTM(
            input_size=50, hidden_size=128,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.3,
        )

        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),   # raw logit
        )

    def forward(self, x):
        feat = self.cnn(x).permute(0, 2, 1)
        out, _ = self.lstm(feat)
        centre = out[:, out.shape[1] // 2, :]
        return self.fc(centre).squeeze(1)


# ============================================================
# CELL 5: Load Data
# ============================================================
print("Loading training PKL...")
with open(PREPROCESSED + "washing_machine_train.pkl", "rb") as f:
    tr = pickle.load(f)
X_train = tr["X"]
y_train = tr["y_label"]
del tr; gc.collect()

print("Loading val PKL...")
with open(PREPROCESSED + "washing_machine_val.pkl", "rb") as f:
    vl = pickle.load(f)
X_val_full = vl["X"]
y_val_full = vl["y_label"]
del vl; gc.collect()

# ── Val sample: real-ish distribution (6:1 OFF:ON) ────────
rng      = np.random.default_rng(RANDOM_SEED)
idx_on   = np.where(y_val_full == 1)[0]
idx_off  = np.where(y_val_full == 0)[0]
n_on     = min(N_VAL_ON, len(idx_on))
n_off    = min(n_on * VAL_OFF_MULT, len(idx_off))
s_on     = rng.choice(idx_on,  size=n_on,  replace=False)
s_off    = rng.choice(idx_off, size=n_off, replace=False)
val_idx  = np.sort(np.concatenate([s_on, s_off]))
X_val    = X_val_full[val_idx]
y_val    = y_val_full[val_idx]

on_pct = 100 * n_on / (n_on + n_off)
print(f"✅ Train : {len(X_train):,} windows (50/50 balanced)")
print(f"✅ Val   : {len(val_idx):,} windows | ON={n_on:,} ({on_pct:.1f}%) OFF={n_off:,}")

train_ds = NILMDataset(X_train, y_train)
val_ds   = NILMDataset(X_val,   y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print("✅ DataLoaders ready")


# ============================================================
# CELL 6: Training
# ============================================================
model     = CNNBiLSTM(window=WINDOW_SIZE).to(device)
criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1,
)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

best_f1          = 0.0
patience_counter = 0
save_path        = MODELS_PATH + "washing_machine_cnn_bilstm_best.pt"
EVAL_THRESH      = 0.50

print(f"\n{'='*55}")
print(f" Training CNN-BiLSTM v3 | Washing Machine")
print(f" Epochs={EPOCHS} | BS={BATCH_SIZE} | Patience={PATIENCE}")
print(f" FocalLoss(alpha=0.75, gamma=2) | Val ratio 1:{VAL_OFF_MULT}")
print(f"{'='*55}\n")

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss += loss.item() * len(y_b)
    train_loss /= len(train_ds)

    # ── Validate ───────────────────────────────────────────
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b = X_b.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(X_b)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y_b.numpy())

    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds  = (probs >= EVAL_THRESH).astype(int)

    f1   = f1_score(labels, preds, pos_label=1, zero_division=0)
    prec = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec  = recall_score(labels, preds, pos_label=1, zero_division=0)

    flag = ""
    if f1 > best_f1:
        best_f1 = f1
        patience_counter = 0
        torch.save(model.state_dict(), save_path)
        flag = " ← ✅ best saved"
    else:
        patience_counter += 1
        flag = f" (patience {patience_counter}/{PATIENCE})"

    print(f"Ep {epoch:02d}/{EPOCHS} | loss={train_loss:.4f} | "
          f"F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f}{flag}")

    if patience_counter >= PATIENCE:
        print(f"\n⏹ Early stop at epoch {epoch}")
        break

print(f"\n✅ Training complete | Best F1 = {best_f1:.4f}")
print(f"   Model → {save_path}")


# ============================================================
# CELL 7: Full Evaluation + Threshold Sweep
# ============================================================
model.load_state_dict(
    torch.load(save_path, map_location=device, weights_only=False)
)
model.eval()

print("\nRunning inference on full val set (623,419 windows)...")

full_ds     = NILMDataset(X_val_full, y_val_full)
full_loader = DataLoader(full_ds, batch_size=1024, shuffle=False,
                         num_workers=2, pin_memory=True)

all_probs, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in full_loader:
        X_b = X_b.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(X_b)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(y_b.numpy())

full_probs  = np.concatenate(all_probs)
full_labels = np.concatenate(all_labels)

# ── Threshold Sweep ────────────────────────────────────────
print(f"\n{'Thresh':>7} | {'F1':>6} | {'Prec':>6} | {'Rec':>6} | "
      f"{'TP':>6} | {'FP':>6} | {'FN':>6}")
print("-" * 62)

best_thresh   = 0.50
best_sweep_f1 = 0.0

for thresh in np.arange(0.10, 1.00, 0.05):
    preds = (full_probs >= thresh).astype(int)
    f1    = f1_score(full_labels,  preds, pos_label=1, zero_division=0)
    prec  = precision_score(full_labels, preds, pos_label=1, zero_division=0)
    rec   = recall_score(full_labels,    preds, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(full_labels, preds, labels=[0, 1]).ravel()

    marker = " ←" if f1 > best_sweep_f1 else ""
    if f1 > best_sweep_f1:
        best_sweep_f1 = f1
        best_thresh   = thresh

    print(f"  {thresh:.2f}   | {f1:.4f} | {prec:.4f} | {rec:.4f} | "
          f"{tp:>6,} | {fp:>6,} | {fn:>6,}{marker}")

print(f"\n✅ Best threshold = {best_thresh:.2f} | F1 = {best_sweep_f1:.4f}")

# ── Final report ───────────────────────────────────────────
preds_best = (full_probs >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(full_labels, preds_best, labels=[0, 1]).ravel()
f1_final   = f1_score(full_labels,  preds_best, pos_label=1)
prec_final = precision_score(full_labels, preds_best, pos_label=1)
rec_final  = recall_score(full_labels,    preds_best, pos_label=1)

print(f"\n{'='*55}")
print(f" WASHING MACHINE — Final Results (Full Val 623K)")
print(f"{'='*55}")
print(f"  Threshold : {best_thresh:.2f}")
print(f"  F1  (ON)  : {f1_final:.4f}")
print(f"  Precision : {prec_final:.4f}")
print(f"  Recall    : {rec_final:.4f}")
print(f"  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"  Model     : {save_path}")
print(f"{'='*55}")

# ── Project Summary ────────────────────────────────────────
print(f"\n{'='*55}")
print(f" PROJECT SUMMARY — All Appliances")
print(f"{'='*55}")
print(f"  {'Appliance':<20} {'Arch':<16} {'F1':>6} {'Thresh':>7}")
print(f"  {'-'*52}")
print(f"  {'Kettle':<20} {'CNN-BiLSTM':<16} {'0.63':>6} {'0.99':>7}")
print(f"  {'Fridge':<20} {'CNN-BiLSTM':<16} {'0.53':>6} {'0.30':>7}")
print(f"  {'Microwave':<20} {'DilatedCNN':<16} {'0.24':>6} {'0.65':>7}")
print(f"  {'Washing Machine':<20} {'CNN-BiLSTM':<16} {f1_final:>6.2f} {best_thresh:>7.2f}")
print(f"{'='*55}")

Mounted at /content/drive
✅ Drive mounted
✅ Config ready | device=cuda
Loading training PKL...
Loading val PKL...
✅ Train : 165,692 windows (50/50 balanced)
✅ Val   : 105,483 windows | ON=15,069 (14.3%) OFF=90,414
✅ DataLoaders ready

 Training CNN-BiLSTM v3 | Washing Machine
 Epochs=40 | BS=512 | Patience=8
 FocalLoss(alpha=0.75, gamma=2) | Val ratio 1:6

Ep 01/40 | loss=0.0576 | F1=0.4637 Prec=0.3060 Rec=0.9565 ← ✅ best saved
Ep 02/40 | loss=0.0450 | F1=0.3781 Prec=0.2342 Rec=0.9810 (patience 1/8)
Ep 03/40 | loss=0.0459 | F1=0.5354 Prec=0.3739 Rec=0.9423 ← ✅ best saved
Ep 04/40 | loss=0.0443 | F1=0.4695 Prec=0.3093 Rec=0.9734 (patience 1/8)
Ep 05/40 | loss=0.0411 | F1=0.5852 Prec=0.4494 Rec=0.8387 ← ✅ best saved
Ep 06/40 | loss=0.0355 | F1=0.4030 Prec=0.2627 Rec=0.8650 (patience 1/8)
Ep 07/40 | loss=0.0294 | F1=0.6370 Prec=0.4899 Rec=0.9105 ← ✅ best saved
Ep 08/40 | loss=0.0261 | F1=0.4276 Prec=0.2830 Rec=0.8744 (patience 1/8)
Ep 09/40 | loss=0.0216 | F1=0.6628 Prec=0.5678 Rec=0.7960

In [ ]:
# ============================================================
# Washing Machine — Rebuild PKL (Houses 5, 9, 11 Only) + Retrain
# NILM Project | KASIT - University of Jordan
# Supervised by Dr. Ruba Obiedat
# ============================================================
# Reason: houses 2 and 3 have very weak washing machine usage
# House 2: ON=180,916 out of 675K → only 0.27% (more noise than signal)
# House 3: same issue
# Solution: train on houses 5, 9, 11 only — clear and sufficient signal
# ============================================================


# ============================================================
# CELL 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (f1_score, precision_score,
                             recall_score, confusion_matrix)

BASE_PATH    = "/content/drive/MyDrive/new obada/"
RAW_DATA     = BASE_PATH + "CLEAN_REFIT_081116/"
LABELED_PATH = BASE_PATH + "data/labeled/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODELS_PATH  = BASE_PATH + "models/"
METERS_JSON  = BASE_PATH + "confirmed_meters.json"

os.makedirs(PREPROCESSED, exist_ok=True)
os.makedirs(MODELS_PATH,  exist_ok=True)

# Training houses — excluding 2 and 3
TRAIN_HOUSES  = [5, 9, 11]
VAL_HOUSE     = 20
APPLIANCE     = "washing_machine"
WINDOW_SIZE   = 599
RESAMPLE_RULE = "1T"
GAP_FILL_LIMIT = 10
ON_THRESHOLD  = 20
MIN_ON_DUR    = 5      # minutes
STRIDE_TRAIN  = 2
STRIDE_VAL    = 1
RANDOM_SEED   = 42
CHUNK_SIZE    = 50_000

with open(METERS_JSON, "r") as f:
    METER_MAP = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"✅ Config | device={device}")
print(f"   Train houses: {TRAIN_HOUSES} (removed 2 & 3)")
print(f"   Val house   : {VAL_HOUSE}")


# ============================================================
# CELL 3: Rebuild Labeled CSV
# ============================================================
def load_and_resample(house_id):
    path = RAW_DATA + f"CLEAN_House{house_id}.csv"
    df   = pd.read_csv(path, header=0,
                       parse_dates={"timestamp": [0]},
                       infer_datetime_format=True,
                       index_col=False)
    df = df.set_index("timestamp").sort_index()
    if "Unix" in df.columns:
        df = df.drop(columns=["Unix"])
    return df.resample(RESAMPLE_RULE).mean()

def fill_gaps(series):
    return series.fillna(method="ffill", limit=GAP_FILL_LIMIT).clip(lower=0)

def compute_labels(series):
    raw    = (series >= ON_THRESHOLD).astype(int)
    labels = raw.copy()
    in_on, start = False, None
    for i in range(len(labels)):
        if raw.iloc[i] == 1 and not in_on:
            in_on, start = True, i
        elif raw.iloc[i] == 0 and in_on:
            if (i - start) < MIN_ON_DUR:
                labels.iloc[start:i] = 0
            in_on = False
    if in_on and (len(labels) - start) < MIN_ON_DUR:
        labels.iloc[start:] = 0
    return labels

output_csv = LABELED_PATH + f"{APPLIANCE}_h5_9_11.csv"

if os.path.exists(output_csv):
    print(f"⏭ Labeled CSV already exists — skipping rebuild")
else:
    print(f"\nBuilding labeled CSV for houses {TRAIN_HOUSES + [VAL_HOUSE]}...")
    all_rows = []
    for house_id in TRAIN_HOUSES + [VAL_HOUSE]:
        house_key = str(house_id)
        col_name  = METER_MAP[house_key][APPLIANCE]
        print(f"  House {house_id} → column: {col_name}")
        df  = load_and_resample(house_id)
        agg = fill_gaps(df["Aggregate"])
        app = fill_gaps(df[col_name])
        lbl = compute_labels(app)
        hdf = pd.DataFrame({
            "timestamp"              : agg.index,
            "house_id"               : house_id,
            "aggregate_watts"        : agg.values,
            f"{APPLIANCE}_watts"     : app.values,
            f"{APPLIANCE}_label"     : lbl.values,
        }).dropna()
        on  = int(lbl.sum())
        off = int((lbl == 0).sum())
        print(f"    Rows={len(hdf):,} ON={on:,} ({100*on/(on+off):.1f}%) OFF={off:,}")
        all_rows.append(hdf)

    final = pd.concat(all_rows, ignore_index=True)
    final.to_csv(output_csv, index=False)
    total_on  = int(final[f"{APPLIANCE}_label"].sum())
    total_off = int((final[f"{APPLIANCE}_label"] == 0).sum())
    print(f"\n✅ Saved: {os.path.basename(output_csv)}")
    print(f"   Total rows={len(final):,} ON={total_on:,} ({100*total_on/(total_on+total_off):.1f}%)")


# ============================================================
# CELL 4: Extract Windows → PKL
# ============================================================
train_pkl = PREPROCESSED + f"{APPLIANCE}_train_h5911.pkl"
val_pkl   = PREPROCESSED + f"{APPLIANCE}_val_h5911.pkl"

def extract_windows(agg, app_watts, app_labels, stride):
    half = WINDOW_SIZE // 2
    X, yw, yl = [], [], []
    n = len(agg)
    if n < WINDOW_SIZE:
        return np.array([]), np.array([]), np.array([])
    for start in range(0, n - WINDOW_SIZE + 1, stride):
        end    = start + WINDOW_SIZE
        centre = start + half
        window = agg[start:end]
        if np.any(np.isnan(window)) or np.isnan(app_watts[centre]):
            continue
        X.append(window)
        yw.append(app_watts[centre])
        yl.append(app_labels[centre])
    if not X:
        return np.array([]), np.array([]), np.array([])
    return (np.array(X,  dtype=np.float32),
            np.array(yw, dtype=np.float32),
            np.array(yl, dtype=np.int8))

def build_pkl(houses, split, output_path, do_balance=True):
    if os.path.exists(output_path):
        print(f"⏭ {os.path.basename(output_path)} exists — skipping")
        return
    stride   = STRIDE_TRAIN if split == "train" else STRIDE_VAL
    all_X, all_yw, all_yl = [], [], []
    overlap  = WINDOW_SIZE - 1
    for house_id in houses:
        print(f"  Windowing house {house_id}...")
        leftover = pd.DataFrame()
        reader   = pd.read_csv(output_csv, chunksize=CHUNK_SIZE,
                               parse_dates=["timestamp"])
        for raw_chunk in reader:
            chunk = raw_chunk[raw_chunk["house_id"] == house_id].copy()
            if len(chunk) == 0:
                continue
            if len(leftover) > 0:
                chunk = pd.concat([leftover, chunk], ignore_index=True)
            leftover = chunk.tail(overlap).copy()
            if len(chunk) < WINDOW_SIZE:
                continue
            agg_v = chunk["aggregate_watts"].values.astype(np.float32)
            mn, mx = np.nanmin(agg_v), np.nanmax(agg_v)
            if mx - mn < 1e-6:
                continue
            agg_n  = (agg_v - mn) / (mx - mn)
            app_w  = chunk[f"{APPLIANCE}_watts"].values.astype(np.float32)
            app_l  = chunk[f"{APPLIANCE}_label"].values.astype(np.int8)
            X, yw, yl = extract_windows(agg_n, app_w, app_l, stride)
            if len(X) == 0:
                continue
            all_X.append(X); all_yw.append(yw); all_yl.append(yl)
            del X, yw, yl, agg_n; gc.collect()

    X  = np.concatenate(all_X,  axis=0)
    yw = np.concatenate(all_yw, axis=0)
    yl = np.concatenate(all_yl, axis=0)
    del all_X, all_yw, all_yl; gc.collect()

    if do_balance:
        rng_l   = np.random.default_rng(RANDOM_SEED)
        on_idx  = np.where(yl == 1)[0]
        off_idx = np.where(yl == 0)[0]
        sampled = rng_l.choice(off_idx, size=min(len(on_idx), len(off_idx)),
                               replace=False)
        keep    = np.sort(np.concatenate([on_idx, sampled]))
        X, yw, yl = X[keep], yw[keep], yl[keep]

    with open(output_path, "wb") as f:
        pickle.dump({"X": X, "y_watts": yw, "y_label": yl}, f, protocol=4)

    n_on  = int(yl.sum())
    n_off = int((yl == 0).sum())
    size  = os.path.getsize(output_path) / 1e6
    print(f"  ✅ {os.path.basename(output_path)}: "
          f"{len(X):,} windows | ON={n_on:,} OFF={n_off:,} | {size:.1f} MB")

print("\nExtracting windows...")
build_pkl(TRAIN_HOUSES, "train", train_pkl, do_balance=True)
build_pkl([VAL_HOUSE],  "val",   val_pkl,   do_balance=False)
print("✅ PKLs ready")


# ============================================================
# CELL 5: Load Data
# ============================================================
print("\nLoading PKLs into memory...")
with open(train_pkl, "rb") as f:
    tr = pickle.load(f)
X_train = tr["X"];  y_train = tr["y_label"]
del tr; gc.collect()

with open(val_pkl, "rb") as f:
    vl = pickle.load(f)
X_val_full = vl["X"];  y_val_full = vl["y_label"]
del vl; gc.collect()

# Val sample: 1:6 ratio
rng     = np.random.default_rng(RANDOM_SEED)
idx_on  = np.where(y_val_full == 1)[0]
idx_off = np.where(y_val_full == 0)[0]
n_on    = len(idx_on)
n_off   = min(n_on * 6, len(idx_off))
s_on    = idx_on
s_off   = rng.choice(idx_off, size=n_off, replace=False)
val_idx = np.sort(np.concatenate([s_on, s_off]))
X_val   = X_val_full[val_idx]
y_val   = y_val_full[val_idx]

on_pct = 100 * n_on / (n_on + n_off)
print(f"✅ Train : {len(X_train):,} windows (50/50)")
print(f"✅ Val   : {len(val_idx):,} | ON={n_on:,} ({on_pct:.1f}%) OFF={n_off:,}")


# ============================================================
# CELL 6: Model + Training
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
                  logits, targets, reduction='none')
        p_t = torch.sigmoid(logits) * targets + \
              (1 - torch.sigmoid(logits)) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

class CNNBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,  30, 10, padding=4), nn.BatchNorm1d(30), nn.ReLU(),
            nn.Conv1d(30, 30,  8, padding=3), nn.BatchNorm1d(30), nn.ReLU(),
            nn.Conv1d(30, 40,  6, padding=2), nn.BatchNorm1d(40), nn.ReLU(),
            nn.Conv1d(40, 50,  5, padding=2), nn.BatchNorm1d(50), nn.ReLU(),
            nn.Conv1d(50, 50,  5, padding=2), nn.BatchNorm1d(50), nn.ReLU(),
        )
        self.lstm = nn.LSTM(50, 128, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.fc   = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )
    def forward(self, x):
        f = self.cnn(x).permute(0, 2, 1)
        o, _ = self.lstm(f)
        return self.fc(o[:, o.shape[1]//2, :]).squeeze(1)

train_ds = NILMDataset(X_train, y_train)
val_ds   = NILMDataset(X_val,   y_val)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)

model     = CNNBiLSTM().to(device)
criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=40, pct_start=0.1)
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type=='cuda'))

best_f1, patience_ctr = 0.0, 0
PATIENCE  = 8
save_path = MODELS_PATH + "washing_machine_cnn_bilstm_best.pt"

print(f"\n{'='*55}")
print(f" Training | Houses 5,9,11 only | FocalLoss | OneCycleLR")
print(f"{'='*55}\n")

for epoch in range(1, 41):
    model.train()
    tr_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            loss = criterion(model(Xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    model.eval()
    probs_l, labs_l = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
                logits = model(Xb.to(device))
            probs_l.append(torch.sigmoid(logits).cpu().numpy())
            labs_l.append(yb.numpy())
    probs  = np.concatenate(probs_l)
    labels = np.concatenate(labs_l)
    preds  = (probs >= 0.50).astype(int)
    f1   = f1_score(labels, preds, pos_label=1, zero_division=0)
    prec = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec  = recall_score(labels, preds, pos_label=1, zero_division=0)

    flag = ""
    if f1 > best_f1:
        best_f1 = f1; patience_ctr = 0
        torch.save(model.state_dict(), save_path)
        flag = " ← ✅ best saved"
    else:
        patience_ctr += 1
        flag = f" (patience {patience_ctr}/{PATIENCE})"

    print(f"Ep {epoch:02d}/40 | loss={tr_loss:.4f} | "
          f"F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f}{flag}")
    if patience_ctr >= PATIENCE:
        print(f"\n⏹ Early stop at epoch {epoch}"); break

print(f"\n✅ Done | Best F1 = {best_f1:.4f} | Model → {save_path}")


# ============================================================
# CELL 7: Full Evaluation + Threshold Sweep
# ============================================================
model.load_state_dict(
    torch.load(save_path, map_location=device, weights_only=False))
model.eval()

print("\nInference on full val (623,419 windows)...")
full_ds     = NILMDataset(X_val_full, y_val_full)
full_loader = DataLoader(full_ds, batch_size=1024, shuffle=False,
                         num_workers=2, pin_memory=True)

all_p, all_l = [], []
with torch.no_grad():
    for Xb, yb in full_loader:
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            logits = model(Xb.to(device))
        all_p.append(torch.sigmoid(logits).cpu().numpy())
        all_l.append(yb.numpy())

full_probs  = np.concatenate(all_p)
full_labels = np.concatenate(all_l)

print(f"\n{'Thresh':>7} | {'F1':>6} | {'Prec':>6} | {'Rec':>6} | "
      f"{'TP':>6} | {'FP':>6} | {'FN':>6}")
print("-" * 62)

best_thresh, best_f1_sw = 0.50, 0.0
for thresh in np.arange(0.10, 1.00, 0.05):
    preds = (full_probs >= thresh).astype(int)
    f1    = f1_score(full_labels,  preds, pos_label=1, zero_division=0)
    prec  = precision_score(full_labels, preds, pos_label=1, zero_division=0)
    rec   = recall_score(full_labels,    preds, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(full_labels, preds, labels=[0,1]).ravel()
    marker = " ←" if f1 > best_f1_sw else ""
    if f1 > best_f1_sw:
        best_f1_sw = f1; best_thresh = thresh
    print(f"  {thresh:.2f}   | {f1:.4f} | {prec:.4f} | {rec:.4f} | "
          f"{tp:>6,} | {fp:>6,} | {fn:>6,}{marker}")

pb = (full_probs >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(full_labels, pb, labels=[0,1]).ravel()
f1f  = f1_score(full_labels, pb, pos_label=1)
prf  = precision_score(full_labels, pb, pos_label=1)
rcf  = recall_score(full_labels, pb, pos_label=1)

print(f"\n{'='*55}")
print(f" WASHING MACHINE — Final Results (Full Val 623K)")
print(f"{'='*55}")
print(f"  Threshold : {best_thresh:.2f}")
print(f"  F1  (ON)  : {f1f:.4f}")
print(f"  Precision : {prf:.4f}")
print(f"  Recall    : {rcf:.4f}")
print(f"  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"  Model     : {save_path}")
print(f"{'='*55}")

print(f"\n{'='*55}")
print(f" PROJECT SUMMARY — All Appliances")
print(f"{'='*55}")
print(f"  {'Appliance':<20} {'Arch':<16} {'F1':>6} {'Thresh':>7}")
print(f"  {'-'*52}")
print(f"  {'Kettle':<20} {'CNN-BiLSTM':<16} {'0.63':>6} {'0.99':>7}")
print(f"  {'Fridge':<20} {'CNN-BiLSTM':<16} {'0.53':>6} {'0.30':>7}")
print(f"  {'Microwave':<20} {'DilatedCNN':<16} {'0.24':>6} {'0.65':>7}")
print(f"  {'Washing Machine':<20} {'CNN-BiLSTM':<16} {f1f:>6.2f} {best_thresh:>7.2f}")
print(f"{'='*55}")

Mounted at /content/drive
✅ Drive mounted
✅ Config | device=cuda
   Train houses: [5, 9, 11] (removed 2 & 3)
   Val house   : 20

Building labeled CSV for houses [5, 9, 11, 20]...
  House 5 → column: Appliance3


/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:80: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  return df.resample(RESAMPLE_RULE).mean()
/tmp/ipykernel_1467/2678085966.py:83: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return series.fillna(method="ffill", limit=GAP_FILL_LIMIT).clip(lowe

    Rows=855,877 ON=65,903 (7.1%) OFF=867,690
  House 9 → column: Appliance3


/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:80: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  return df.resample(RESAMPLE_RULE).mean()
/tmp/ipykernel_1467/2678085966.py:83: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return series.fillna(method="ffill", limit=GAP_FILL_LIMIT).clip(lowe

    Rows=693,704 ON=18,531 (2.3%) OFF=799,476
  House 11 → column: Appliance3


/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:80: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  return df.resample(RESAMPLE_RULE).mean()
/tmp/ipykernel_1467/2678085966.py:83: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return series.fillna(method="ffill", limit=GAP_FILL_LIMIT).clip(lowe

    Rows=503,953 ON=6,629 (1.2%) OFF=558,068
  House 20 → column: Appliance4


/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:73: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df   = pd.read_csv(path, header=0,
/tmp/ipykernel_1467/2678085966.py:80: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  return df.resample(RESAMPLE_RULE).mean()
/tmp/ipykernel_1467/2678085966.py:83: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  return series.fillna(method="ffill", limit=GAP_FILL_LIMIT).clip(lowe

    Rows=624,017 ON=15,069 (2.3%) OFF=647,795

✅ Saved: washing_machine_h5_9_11.csv
   Total rows=2,677,551 ON=106,132 (4.0%)

Extracting windows...
  Windowing house 5...
  Windowing house 9...
  Windowing house 11...
  ✅ washing_machine_train_h5911.pkl: 91,038 windows | ON=45,519 OFF=45,519 | 218.6 MB
  Windowing house 20...
  ✅ washing_machine_val_h5911.pkl: 623,419 windows | ON=15,069 OFF=608,350 | 1496.8 MB
✅ PKLs ready

Loading PKLs into memory...
✅ Train : 91,038 windows (50/50)
✅ Val   : 105,483 | ON=15,069 (14.3%) OFF=90,414

 Training | Houses 5,9,11 only | FocalLoss | OneCycleLR

Ep 01/40 | loss=0.0631 | F1=0.3118 Prec=0.1862 Rec=0.9567 ← ✅ best saved
Ep 02/40 | loss=0.0514 | F1=0.3086 Prec=0.1833 Rec=0.9753 (patience 1/8)
Ep 03/40 | loss=0.0496 | F1=0.2491 Prec=0.1424 Rec=0.9930 (patience 2/8)
Ep 04/40 | loss=0.0461 | F1=0.2500 Prec=0.1429 Rec=1.0000 (patience 3/8)
Ep 05/40 | loss=0.0449 | F1=0.4076 Prec=0.2625 Rec=0.9119 ← ✅ best saved
Ep 06/40 | loss=0.0454 | F1=0.2641 Pr

In [ ]:
# ============================================================
# Washing Machine — CNN-BiLSTM Seq2Point v5
# NILM Project | KASIT - University of Jordan
# Supervised by Dr. Ruba Obiedat
# ============================================================
# Strategy:
#   - Use all houses (2,3,5,9,11) with smart sampling
#   - House 5: all ON windows (65K) — richest source
#   - Houses 9,11: all ON windows
#   - Houses 2,3: max 5000 ON windows each
#     (capture signal without drowning in noise)
#   - OFF windows = same count as total ON (50/50)
# ============================================================


# ============================================================
# CELL 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (f1_score, precision_score,
                             recall_score, confusion_matrix)

BASE_PATH    = "/content/drive/MyDrive/new obada/"
RAW_DATA     = BASE_PATH + "CLEAN_REFIT_081116/"
LABELED_PATH = BASE_PATH + "data/labeled/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODELS_PATH  = BASE_PATH + "models/"
METERS_JSON  = BASE_PATH + "confirmed_meters.json"

os.makedirs(PREPROCESSED, exist_ok=True)
os.makedirs(MODELS_PATH,  exist_ok=True)

ALL_TRAIN_HOUSES = [2, 3, 5, 9, 11]
VAL_HOUSE        = 20
APPLIANCE        = "washing_machine"
WINDOW_SIZE      = 599
RESAMPLE_RULE    = "1min"
GAP_FILL_LIMIT   = 10
ON_THRESHOLD     = 20
MIN_ON_DUR       = 5
STRIDE_TRAIN     = 2
STRIDE_VAL       = 1
RANDOM_SEED      = 42
CHUNK_SIZE       = 50_000

# Cap ON windows from weak houses
WEAK_HOUSES      = {2: 5000, 3: 5000}   # max ON windows per weak house

with open(METERS_JSON, "r") as f:
    METER_MAP = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"✅ Config | device={device}")
print(f"   Strong houses: 5, 9, 11 (all ON windows)")
print(f"   Weak houses  : 2, 3 (max {WEAK_HOUSES[2]:,} ON each)")


# ============================================================
# CELL 3: Helper Functions
# ============================================================
def load_and_resample(house_id):
    path = RAW_DATA + f"CLEAN_House{house_id}.csv"
    df   = pd.read_csv(path, header=0, index_col=False)
    # parse timestamp manually (avoids deprecation warnings)
    df.columns = ["timestamp"] + list(df.columns[1:])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()
    if "Unix" in df.columns:
        df = df.drop(columns=["Unix"])
    return df.resample(RESAMPLE_RULE).mean()

def fill_gaps(series):
    return series.ffill(limit=GAP_FILL_LIMIT).clip(lower=0)

def compute_labels(series):
    raw    = (series >= ON_THRESHOLD).astype(int)
    labels = raw.copy()
    in_on, start = False, None
    for i in range(len(labels)):
        if raw.iloc[i] == 1 and not in_on:
            in_on, start = True, i
        elif raw.iloc[i] == 0 and in_on:
            if (i - start) < MIN_ON_DUR:
                labels.iloc[start:i] = 0
            in_on = False
    if in_on and (len(labels) - start) < MIN_ON_DUR:
        labels.iloc[start:] = 0
    return labels

print("✅ Helpers defined")


# ============================================================
# CELL 4: Build Labeled CSV (all houses + val)
# ============================================================
output_csv = LABELED_PATH + f"{APPLIANCE}_all5.csv"

if os.path.exists(output_csv):
    print(f"⏭ Labeled CSV exists — skipping")
else:
    print("\nBuilding labeled CSV...")
    all_rows = []
    for house_id in ALL_TRAIN_HOUSES + [VAL_HOUSE]:
        house_key = str(house_id)
        col_name  = METER_MAP[house_key][APPLIANCE]
        print(f"  House {house_id} → {col_name}")
        df  = load_and_resample(house_id)
        agg = fill_gaps(df["Aggregate"])
        app = fill_gaps(df[col_name])
        lbl = compute_labels(app)
        hdf = pd.DataFrame({
            "timestamp"          : agg.index,
            "house_id"           : house_id,
            "aggregate_watts"    : agg.values,
            f"{APPLIANCE}_watts" : app.values,
            f"{APPLIANCE}_label" : lbl.values,
        }).dropna()
        on  = int(lbl.sum())
        off = int((lbl == 0).sum())
        print(f"    rows={len(hdf):,} ON={on:,} ({100*on/(on+off):.1f}%)")
        all_rows.append(hdf)

    final = pd.concat(all_rows, ignore_index=True)
    final.to_csv(output_csv, index=False)
    print(f"\n✅ Saved: {os.path.basename(output_csv)} | {len(final):,} rows")


# ============================================================
# CELL 5: Extract Windows → Smart PKL
# ============================================================
train_pkl = PREPROCESSED + f"{APPLIANCE}_train_v5.pkl"
val_pkl   = PREPROCESSED + f"{APPLIANCE}_val_v5.pkl"

def extract_windows_from_chunk(agg, app_watts, app_labels, stride):
    half = WINDOW_SIZE // 2
    X, yw, yl = [], [], []
    n = len(agg)
    if n < WINDOW_SIZE:
        return np.array([]), np.array([]), np.array([])
    for start in range(0, n - WINDOW_SIZE + 1, stride):
        end    = start + WINDOW_SIZE
        centre = start + half
        window = agg[start:end]
        if np.any(np.isnan(window)) or np.isnan(app_watts[centre]):
            continue
        X.append(window)
        yw.append(app_watts[centre])
        yl.append(app_labels[centre])
    if not X:
        return np.array([]), np.array([]), np.array([])
    return (np.array(X,  dtype=np.float32),
            np.array(yw, dtype=np.float32),
            np.array(yl, dtype=np.int8))

def build_train_pkl_smart(output_path):
    """
  Extract windows per house separately
 Weak houses: cap ON windows + equal OFF count
 Strong houses: standard 50/50
 Then merge all """
    if os.path.exists(output_path):
        print(f"⏭ {os.path.basename(output_path)} exists — skipping")
        return

    rng = np.random.default_rng(RANDOM_SEED)
    all_X, all_yw, all_yl = [], [], []

    for house_id in ALL_TRAIN_HOUSES:
        print(f"\n  House {house_id}:")
        leftover = pd.DataFrame()
        hX, hyw, hyl = [], [], []
        overlap  = WINDOW_SIZE - 1
        reader   = pd.read_csv(output_csv, chunksize=CHUNK_SIZE,
                               parse_dates=["timestamp"])
        for raw_chunk in reader:
            chunk = raw_chunk[raw_chunk["house_id"] == house_id].copy()
            if len(chunk) == 0:
                continue
            if len(leftover) > 0:
                chunk = pd.concat([leftover, chunk], ignore_index=True)
            leftover = chunk.tail(overlap).copy()
            if len(chunk) < WINDOW_SIZE:
                continue
            agg_v = chunk["aggregate_watts"].values.astype(np.float32)
            mn, mx = np.nanmin(agg_v), np.nanmax(agg_v)
            if mx - mn < 1e-6:
                continue
            agg_n = (agg_v - mn) / (mx - mn)
            app_w = chunk[f"{APPLIANCE}_watts"].values.astype(np.float32)
            app_l = chunk[f"{APPLIANCE}_label"].values.astype(np.int8)
            X, yw, yl = extract_windows_from_chunk(agg_n, app_w, app_l,
                                                    STRIDE_TRAIN)
            if len(X) == 0:
                continue
            hX.append(X); hyw.append(yw); hyl.append(yl)
            del X, yw, yl, agg_n; gc.collect()

        if not hX:
            print(f"    ⚠ No windows found")
            continue

        hX  = np.concatenate(hX,  axis=0)
        hyw = np.concatenate(hyw, axis=0)
        hyl = np.concatenate(hyl, axis=0)

        on_idx  = np.where(hyl == 1)[0]
        off_idx = np.where(hyl == 0)[0]
        n_on    = len(on_idx)

       # Weak houses: cap ON windows
        if house_id in WEAK_HOUSES:
            cap = WEAK_HOUSES[house_id]
            if n_on > cap:
                on_idx = rng.choice(on_idx, size=cap, replace=False)
                n_on   = cap

       # OFF = same count as ON
        n_off   = min(n_on, len(off_idx))
        off_sel = rng.choice(off_idx, size=n_off, replace=False)
        keep    = np.sort(np.concatenate([on_idx, off_sel]))
        hX, hyw, hyl = hX[keep], hyw[keep], hyl[keep]

        print(f"    windows kept: {len(hX):,} | ON={n_on:,} OFF={n_off:,}")
        all_X.append(hX); all_yw.append(hyw); all_yl.append(hyl)
        del hX, hyw, hyl; gc.collect()

    X  = np.concatenate(all_X,  axis=0)
    yw = np.concatenate(all_yw, axis=0)
    yl = np.concatenate(all_yl, axis=0)

    # Shuffle
    idx = rng.permutation(len(X))
    X, yw, yl = X[idx], yw[idx], yl[idx]

    with open(output_path, "wb") as f:
        pickle.dump({"X": X, "y_watts": yw, "y_label": yl}, f, protocol=4)

    n_on  = int(yl.sum())
    n_off = int((yl == 0).sum())
    size  = os.path.getsize(output_path) / 1e6
    print(f"\n  ✅ {os.path.basename(output_path)}: "
          f"{len(X):,} windows | ON={n_on:,} OFF={n_off:,} | {size:.1f} MB")

def build_val_pkl(output_path):
    if os.path.exists(output_path):
        print(f"⏭ {os.path.basename(output_path)} exists — skipping")
        return
    print(f"\n  Windowing val house {VAL_HOUSE}...")
    leftover = pd.DataFrame()
    all_X, all_yw, all_yl = [], [], []
    overlap  = WINDOW_SIZE - 1
    reader   = pd.read_csv(output_csv, chunksize=CHUNK_SIZE,
                            parse_dates=["timestamp"])
    for raw_chunk in reader:
        chunk = raw_chunk[raw_chunk["house_id"] == VAL_HOUSE].copy()
        if len(chunk) == 0:
            continue
        if len(leftover) > 0:
            chunk = pd.concat([leftover, chunk], ignore_index=True)
        leftover = chunk.tail(overlap).copy()
        if len(chunk) < WINDOW_SIZE:
            continue
        agg_v = chunk["aggregate_watts"].values.astype(np.float32)
        mn, mx = np.nanmin(agg_v), np.nanmax(agg_v)
        if mx - mn < 1e-6:
            continue
        agg_n = (agg_v - mn) / (mx - mn)
        app_w = chunk[f"{APPLIANCE}_watts"].values.astype(np.float32)
        app_l = chunk[f"{APPLIANCE}_label"].values.astype(np.int8)
        X, yw, yl = extract_windows_from_chunk(agg_n, app_w, app_l, STRIDE_VAL)
        if len(X) == 0:
            continue
        all_X.append(X); all_yw.append(yw); all_yl.append(yl)
        del X, yw, yl; gc.collect()

    X  = np.concatenate(all_X,  axis=0)
    yw = np.concatenate(all_yw, axis=0)
    yl = np.concatenate(all_yl, axis=0)
    with open(output_path, "wb") as f:
        pickle.dump({"X": X, "y_watts": yw, "y_label": yl}, f, protocol=4)
    n_on = int(yl.sum())
    size = os.path.getsize(output_path) / 1e6
    print(f"  ✅ {os.path.basename(output_path)}: "
          f"{len(X):,} windows | ON={n_on:,} | {size:.1f} MB")

print("Extracting windows (smart sampling)...")
build_train_pkl_smart(train_pkl)
build_val_pkl(val_pkl)
print("\n✅ PKLs ready")


# ============================================================
# CELL 6: Load Data
# ============================================================
print("\nLoading PKLs...")
with open(train_pkl, "rb") as f:
    tr = pickle.load(f)
X_train = tr["X"];  y_train = tr["y_label"]
del tr; gc.collect()

with open(val_pkl, "rb") as f:
    vl = pickle.load(f)
X_val_full = vl["X"];  y_val_full = vl["y_label"]
del vl; gc.collect()

# Val sample 1:6
rng     = np.random.default_rng(RANDOM_SEED)
idx_on  = np.where(y_val_full == 1)[0]
idx_off = np.where(y_val_full == 0)[0]
n_on    = len(idx_on)
n_off   = min(n_on * 6, len(idx_off))
s_off   = rng.choice(idx_off, size=n_off, replace=False)
val_idx = np.sort(np.concatenate([idx_on, s_off]))
X_val   = X_val_full[val_idx]
y_val   = y_val_full[val_idx]

print(f"✅ Train : {len(X_train):,} windows")
print(f"✅ Val   : {len(val_idx):,} | ON={n_on:,} ({100*n_on/(n_on+n_off):.1f}%) OFF={n_off:,}")


# ============================================================
# CELL 7: Model + Training
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
                  logits, targets, reduction='none')
        p_t = torch.sigmoid(logits) * targets + \
              (1 - torch.sigmoid(logits)) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

class CNNBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,  30, 10, padding=4), nn.BatchNorm1d(30), nn.ReLU(),
            nn.Conv1d(30, 30,  8, padding=3), nn.BatchNorm1d(30), nn.ReLU(),
            nn.Conv1d(30, 40,  6, padding=2), nn.BatchNorm1d(40), nn.ReLU(),
            nn.Conv1d(40, 50,  5, padding=2), nn.BatchNorm1d(50), nn.ReLU(),
            nn.Conv1d(50, 50,  5, padding=2), nn.BatchNorm1d(50), nn.ReLU(),
        )
        self.lstm = nn.LSTM(50, 128, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )
    def forward(self, x):
        f = self.cnn(x).permute(0, 2, 1)
        o, _ = self.lstm(f)
        return self.fc(o[:, o.shape[1]//2, :]).squeeze(1)

train_ds = NILMDataset(X_train, y_train)
val_ds   = NILMDataset(X_val,   y_val)
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)

EPOCHS    = 40
PATIENCE  = 8
model     = CNNBiLSTM().to(device)
criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1)
scaler    = torch.amp.GradScaler('cuda', enabled=(device.type=='cuda'))

best_f1, pat_ctr = 0.0, 0
save_path = MODELS_PATH + "washing_machine_cnn_bilstm_best.pt"

print(f"\n{'='*55}")
print(f" Training v5 | All 5 houses (smart sampling)")
print(f" FocalLoss | OneCycleLR | Patience={PATIENCE}")
print(f"{'='*55}\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            loss = criterion(model(Xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    model.eval()
    probs_l, labs_l = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
                logits = model(Xb.to(device))
            probs_l.append(torch.sigmoid(logits).cpu().numpy())
            labs_l.append(yb.numpy())
    probs  = np.concatenate(probs_l)
    labels = np.concatenate(labs_l)
    preds  = (probs >= 0.50).astype(int)
    f1   = f1_score(labels, preds, pos_label=1, zero_division=0)
    prec = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec  = recall_score(labels, preds, pos_label=1, zero_division=0)

    flag = ""
    if f1 > best_f1:
        best_f1 = f1; pat_ctr = 0
        torch.save(model.state_dict(), save_path)
        flag = " ← ✅ best saved"
    else:
        pat_ctr += 1
        flag = f" (patience {pat_ctr}/{PATIENCE})"

    print(f"Ep {epoch:02d}/{EPOCHS} | loss={tr_loss:.4f} | "
          f"F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f}{flag}")
    if pat_ctr >= PATIENCE:
        print(f"\n⏹ Early stop at epoch {epoch}"); break

print(f"\n✅ Done | Best F1 = {best_f1:.4f} | Model → {save_path}")


# ============================================================
# CELL 8: Full Evaluation + Threshold Sweep
# ============================================================
model.load_state_dict(
    torch.load(save_path, map_location=device, weights_only=False))
model.eval()

print("\nInference on full val (623,419 windows)...")
full_ds     = NILMDataset(X_val_full, y_val_full)
full_loader = DataLoader(full_ds, batch_size=1024, shuffle=False,
                         num_workers=2, pin_memory=True)

all_p, all_l = [], []
with torch.no_grad():
    for Xb, yb in full_loader:
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            logits = model(Xb.to(device))
        all_p.append(torch.sigmoid(logits).cpu().numpy())
        all_l.append(yb.numpy())

full_probs  = np.concatenate(all_p)
full_labels = np.concatenate(all_l)

print(f"\n{'Thresh':>7} | {'F1':>6} | {'Prec':>6} | {'Rec':>6} | "
      f"{'TP':>6} | {'FP':>6} | {'FN':>6}")
print("-" * 62)

best_thresh, best_f1_sw = 0.50, 0.0
for thresh in np.arange(0.10, 1.00, 0.05):
    preds = (full_probs >= thresh).astype(int)
    f1    = f1_score(full_labels,  preds, pos_label=1, zero_division=0)
    prec  = precision_score(full_labels, preds, pos_label=1, zero_division=0)
    rec   = recall_score(full_labels,    preds, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(full_labels, preds, labels=[0,1]).ravel()
    marker = " ←" if f1 > best_f1_sw else ""
    if f1 > best_f1_sw:
        best_f1_sw = f1; best_thresh = thresh
    print(f"  {thresh:.2f}   | {f1:.4f} | {prec:.4f} | {rec:.4f} | "
          f"{tp:>6,} | {fp:>6,} | {fn:>6,}{marker}")

pb = (full_probs >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(full_labels, pb, labels=[0,1]).ravel()
f1f  = f1_score(full_labels, pb, pos_label=1)
prf  = precision_score(full_labels, pb, pos_label=1)
rcf  = recall_score(full_labels, pb, pos_label=1)

print(f"\n{'='*55}")
print(f" WASHING MACHINE — Final Results (Full Val 623K)")
print(f"{'='*55}")
print(f"  Threshold : {best_thresh:.2f}")
print(f"  F1  (ON)  : {f1f:.4f}")
print(f"  Precision : {prf:.4f}")
print(f"  Recall    : {rcf:.4f}")
print(f"  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"  Model     : {save_path}")
print(f"{'='*55}")

print(f"\n{'='*55}")
print(f" PROJECT SUMMARY — All Appliances")
print(f"{'='*55}")
print(f"  {'Appliance':<20} {'Arch':<16} {'F1':>6} {'Thresh':>7}")
print(f"  {'-'*52}")
print(f"  {'Kettle':<20} {'CNN-BiLSTM':<16} {'0.63':>6} {'0.99':>7}")
print(f"  {'Fridge':<20} {'CNN-BiLSTM':<16} {'0.53':>6} {'0.30':>7}")
print(f"  {'Microwave':<20} {'DilatedCNN':<16} {'0.24':>6} {'0.65':>7}")
print(f"  {'Washing Machine':<20} {'CNN-BiLSTM':<16} {f1f:>6.2f} {best_thresh:>7.2f}")
print(f"{'='*55}")

Mounted at /content/drive
✅ Drive mounted
✅ Config | device=cuda
   Strong houses: 5, 9, 11 (all ON windows)
   Weak houses  : 2, 3 (max 5,000 ON each)
✅ Helpers defined

Building labeled CSV...
  House 2 → Appliance2
    rows=675,227 ON=30,618 (3.4%)
  House 3 → Appliance6
    rows=786,470 ON=44,166 (5.0%)
  House 5 → Appliance3
    rows=855,877 ON=65,903 (7.1%)
  House 9 → Appliance3
    rows=693,704 ON=18,531 (2.3%)
  House 11 → Appliance3
    rows=503,953 ON=6,629 (1.2%)
  House 20 → Appliance4
    rows=624,017 ON=15,069 (2.3%)

✅ Saved: washing_machine_all5.csv | 4,139,248 rows
Extracting windows (smart sampling)...

  House 2:
    windows kept: 10,000 | ON=5,000 OFF=5,000

  House 3:
    windows kept: 10,000 | ON=5,000 OFF=5,000

  House 5:
    windows kept: 65,928 | ON=32,964 OFF=32,964

  House 9:
    windows kept: 18,548 | ON=9,274 OFF=9,274

  House 11:
    windows kept: 6,630 | ON=3,315 OFF=3,315

  ✅ washing_machine_train_v5.pkl: 111,106 windows | ON=55,553 OFF=55,553 | 266

In [ ]:
# ============================================================
# Problem in previous version:
#   LR=9e-5 too small → slow convergence → instability
# Solution:
#   LR=3e-4 with warmup (OneCycleLR)
#   Gradient clipping (max_norm=1.0) → prevents spikes
#   Higher dropout in FC layers
# ============================================================


# ============================================================
# CELL 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (f1_score, precision_score,
                             recall_score, confusion_matrix)

BASE_PATH    = "/content/drive/MyDrive/new obada/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODELS_PATH  = BASE_PATH + "models/"
os.makedirs(MODELS_PATH, exist_ok=True)

TRAIN_PKL = PREPROCESSED + "washing_machine_train_v5.pkl"
VAL_PKL   = PREPROCESSED + "washing_machine_val_v5.pkl"

BATCH_SIZE   = 256
EPOCHS       = 40
PATIENCE     = 8
RANDOM_SEED  = 42
LR           = 3e-4
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.75
LSTM_HIDDEN  = 256
GRAD_CLIP    = 1.0    # Gradient clipping — prevents instability

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"✅ Config | device={device}")
print(f"   LR={LR} | GradClip={GRAD_CLIP} | Focal(γ={FOCAL_GAMMA}, α={FOCAL_ALPHA})")


# ============================================================
# CELL 3: MultiScale + Dataset + Model
# ============================================================
def make_multiscale(X, ma_short=5, ma_long=30):
    N, L = X.shape
    out  = np.zeros((N, 4, L), dtype=np.float32)
    out[:, 0, :] = X
    diff = np.diff(X, axis=1, prepend=X[:, :1])
    d_max = np.abs(diff).max(axis=1, keepdims=True) + 1e-8
    out[:, 1, :] = diff / d_max
    kernel_s = np.ones(ma_short) / ma_short
    kernel_l = np.ones(ma_long)  / ma_long
    for i in range(N):
        out[i, 2, :] = np.convolve(X[i], kernel_s, mode='same')
        out[i, 3, :] = np.convolve(X[i], kernel_l, mode='same')
    return out

class MultiScaleDataset(Dataset):
    def __init__(self, X_ms, y):
        self.X = torch.tensor(X_ms, dtype=torch.float32)
        self.y = torch.tensor(y,    dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
                  logits, targets, reduction='none')
        p_t = torch.sigmoid(logits) * targets + \
              (1 - torch.sigmoid(logits)) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

class MultiScaleCNNBiLSTM(nn.Module):
    def __init__(self, in_channels=4, lstm_hidden=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 16,  10, padding=4),
            nn.BatchNorm1d(16),  nn.ReLU(),
            nn.Conv1d(16,  128, 8, padding=3),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 128, 6, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 128, 5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 128, 5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.lstm = nn.LSTM(128, lstm_hidden, num_layers=1,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),   # higher than previous (was 0.3)
            nn.Linear(128, 1),
        )
    def forward(self, x):
        f = self.cnn(x).permute(0, 2, 1)
        o, _ = self.lstm(f)
        return self.fc(o[:, o.shape[1]//2, :]).squeeze(1)

print("✅ Classes defined")


# ============================================================
# CELL 4: Load & Build MultiScale
# ============================================================
print("Loading PKLs...")
with open(TRAIN_PKL, "rb") as f:
    tr = pickle.load(f)
X_tr_raw = tr["X"];  y_train = tr["y_label"]
del tr; gc.collect()

with open(VAL_PKL, "rb") as f:
    vl = pickle.load(f)
X_vl_raw = vl["X"];  y_val_full = vl["y_label"]
del vl; gc.collect()

print("Building MultiScale (train)...")
X_train_ms = make_multiscale(X_tr_raw); del X_tr_raw; gc.collect()
print("Building MultiScale (val)...")
X_val_ms   = make_multiscale(X_vl_raw); del X_vl_raw; gc.collect()

print(f"✅ Train: {X_train_ms.shape} | Val: {X_val_ms.shape}")

train_ds    = MultiScaleDataset(X_train_ms, y_train)
full_val_ds = MultiScaleDataset(X_val_ms,   y_val_full)

train_loader = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
full_loader  = DataLoader(full_val_ds, batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
print("✅ DataLoaders ready")


# ============================================================
# CELL 5: Training
# ============================================================
model     = MultiScaleCNNBiLSTM(in_channels=4, lstm_hidden=LSTM_HIDDEN).to(device)
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.15,
)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type=='cuda'))

best_f1, pat_ctr = 0.0, 0
save_path = MODELS_PATH + "washing_machine_multiscale_best.pt"

print(f"\n{'='*55}")
print(f" MultiScale v2 | LR={LR} | GradClip={GRAD_CLIP}")
print(f" Full 623K eval every epoch")
print(f"{'='*55}\n")

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────
    model.train()
    tr_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            loss = criterion(model(Xb), yb)
        scaler.scale(loss).backward()
    # Gradient clipping — prevents instability
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    # ── Full 623K Eval ─────────────────────────────────────
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for Xb, yb in full_loader:
            with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
                logits = model(Xb.to(device))
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(yb.numpy())

    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)

    best_ep_f1, best_ep_thresh = 0.0, 0.50
    for t in np.arange(0.30, 0.95, 0.05):
        f = f1_score(labels, (probs >= t).astype(int),
                     pos_label=1, zero_division=0)
        if f > best_ep_f1:
            best_ep_f1 = f; best_ep_thresh = t

    flag = ""
    if best_ep_f1 > best_f1:
        best_f1 = best_ep_f1; pat_ctr = 0
        torch.save({"state_dict": model.state_dict(),
                    "threshold":  best_ep_thresh,
                    "epoch":      epoch}, save_path)
        flag = f" ← ✅ saved (t={best_ep_thresh:.2f})"
    else:
        pat_ctr += 1
        flag = f" (patience {pat_ctr}/{PATIENCE})"

    p_best = (probs >= best_ep_thresh).astype(int)
    prec = precision_score(labels, p_best, pos_label=1, zero_division=0)
    rec  = recall_score(labels,    p_best, pos_label=1, zero_division=0)

    print(f"Ep {epoch:02d}/{EPOCHS} | loss={tr_loss:.4f} | "
          f"F1={best_ep_f1:.4f} Prec={prec:.4f} Rec={rec:.4f} "
          f"t={best_ep_thresh:.2f}{flag}")

    if pat_ctr >= PATIENCE:
        print(f"\n⏹ Early stop at epoch {epoch}"); break

print(f"\n✅ Done | Best F1 = {best_f1:.4f} | Model → {save_path}")


# ============================================================
# CELL 6: Final Threshold Sweep
# ============================================================
ckpt = torch.load(save_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.eval()

print(f"\nBest model: epoch {ckpt['epoch']} | thresh={ckpt['threshold']:.2f}")
print("Final sweep on 623K...\n")

all_probs, all_labels = [], []
with torch.no_grad():
    for Xb, yb in full_loader:
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            logits = model(Xb.to(device))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(yb.numpy())

full_probs  = np.concatenate(all_probs)
full_labels = np.concatenate(all_labels)

print(f"{'Thresh':>7} | {'F1':>6} | {'Prec':>6} | {'Rec':>6} | "
      f"{'TP':>6} | {'FP':>6} | {'FN':>6}")
print("-" * 62)

best_thresh, best_f1_sw = 0.50, 0.0
for thresh in np.arange(0.10, 1.00, 0.05):
    preds = (full_probs >= thresh).astype(int)
    f1    = f1_score(full_labels,  preds, pos_label=1, zero_division=0)
    prec  = precision_score(full_labels, preds, pos_label=1, zero_division=0)
    rec   = recall_score(full_labels,    preds, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(full_labels, preds, labels=[0,1]).ravel()
    marker = " ←" if f1 > best_f1_sw else ""
    if f1 > best_f1_sw:
        best_f1_sw = f1; best_thresh = thresh
    print(f"  {thresh:.2f}   | {f1:.4f} | {prec:.4f} | {rec:.4f} | "
          f"{tp:>6,} | {fp:>6,} | {fn:>6,}{marker}")

pb  = (full_probs >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(full_labels, pb, labels=[0,1]).ravel()
f1f = f1_score(full_labels, pb, pos_label=1)
prf = precision_score(full_labels, pb, pos_label=1)
rcf = recall_score(full_labels,    pb, pos_label=1)

print(f"\n{'='*55}")
print(f" WASHING MACHINE — Final Results (Full Val 623K)")
print(f"{'='*55}")
print(f"  Threshold : {best_thresh:.2f}")
print(f"  F1  (ON)  : {f1f:.4f}")
print(f"  Precision : {prf:.4f}")
print(f"  Recall    : {rcf:.4f}")
print(f"  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"  Model     : {save_path}")
print(f"{'='*55}")

print(f"\n{'='*55}")
print(f" PROJECT SUMMARY — All Appliances")
print(f"{'='*55}")
print(f"  {'Appliance':<20} {'Arch':<20} {'F1':>6} {'Thresh':>7}")
print(f"  {'-'*56}")
print(f"  {'Kettle':<20} {'CNN-BiLSTM':<20} {'0.63':>6} {'0.99':>7}")
print(f"  {'Fridge':<20} {'CNN-BiLSTM':<20} {'0.53':>6} {'0.30':>7}")
print(f"  {'Microwave':<20} {'DilatedCNN':<20} {'0.24':>6} {'0.65':>7}")
print(f"  {'Dishwasher':<20} {'MS-CNN-BiLSTM':<20} {'0.73':>6} {'0.80':>7}")
print(f"  {'Washing Machine':<20} {'MS-CNN-BiLSTM':<20} {f1f:>6.2f} {best_thresh:>7.2f}")
print(f"{'='*55}")

Mounted at /content/drive
✅ Drive mounted
✅ Config | device=cuda
   LR=0.0003 | GradClip=1.0 | Focal(γ=2.0, α=0.75)
✅ Classes defined
Loading PKLs...
Building MultiScale (train)...
Building MultiScale (val)...
✅ Train: (111106, 4, 599) | Val: (623419, 4, 599)
✅ DataLoaders ready

 MultiScale v2 | LR=0.0003 | GradClip=1.0
 Full 623K eval every epoch

Ep 01/40 | loss=0.0603 | F1=0.1998 Prec=0.1186 Rec=0.6353 t=0.65 ← ✅ saved (t=0.65)
Ep 02/40 | loss=0.0504 | F1=0.3404 Prec=0.2523 Rec=0.5231 t=0.70 ← ✅ saved (t=0.70)
Ep 03/40 | loss=0.0475 | F1=0.3917 Prec=0.3391 Rec=0.4637 t=0.80 ← ✅ saved (t=0.80)
Ep 04/40 | loss=0.0442 | F1=0.4080 Prec=0.3197 Rec=0.5635 t=0.80 ← ✅ saved (t=0.80)
Ep 05/40 | loss=0.0399 | F1=0.3443 Prec=0.2585 Rec=0.5153 t=0.80 (patience 1/8)
Ep 06/40 | loss=0.0304 | F1=0.4553 Prec=0.4097 Rec=0.5123 t=0.80 ← ✅ saved (t=0.80)
Ep 07/40 | loss=0.0243 | F1=0.5669 Prec=0.5085 Rec=0.6405 t=0.75 ← ✅ saved (t=0.75)
Ep 08/40 | loss=0.0205 | F1=0.2856 Prec=0.1829 Rec=0.6510 t=0.80

In [ ]:
# ============================================================
# Washing Machine — Retrain v5 (Final)
# Saves model as washing_machine_v5_final.pt
# Unique filename — not overwritten by future experiments
# ============================================================


# ============================================================
# CELL 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (f1_score, precision_score,
                             recall_score, confusion_matrix)

BASE_PATH    = "/content/drive/MyDrive/new obada/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODELS_PATH  = BASE_PATH + "models/"

TRAIN_PKL = PREPROCESSED + "washing_machine_train_v5.pkl"
VAL_PKL   = PREPROCESSED + "washing_machine_val_v5.pkl"

BATCH_SIZE   = 512
EPOCHS       = 40
PATIENCE     = 8
RANDOM_SEED  = 42
N_VAL_ON     = 15_069
VAL_OFF_MULT = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"✅ Config | device={device}")


# ============================================================
# CELL 3: Dataset + Model
# ============================================================
class NILMDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
                  logits, targets, reduction='none')
        p_t = torch.sigmoid(logits) * targets + \
              (1 - torch.sigmoid(logits)) * (1 - targets)
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()

class CNNBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,  30, 10, padding=4), nn.BatchNorm1d(30),  nn.ReLU(),
            nn.Conv1d(30, 30,  8, padding=3), nn.BatchNorm1d(30),  nn.ReLU(),
            nn.Conv1d(30, 40,  6, padding=2), nn.BatchNorm1d(40),  nn.ReLU(),
            nn.Conv1d(40, 50,  5, padding=2), nn.BatchNorm1d(50),  nn.ReLU(),
            nn.Conv1d(50, 50,  5, padding=2), nn.BatchNorm1d(50),  nn.ReLU(),
        )
        self.lstm = nn.LSTM(50, 128, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        f = self.cnn(x).permute(0, 2, 1)
        o, _ = self.lstm(f)
        return self.fc(o[:, o.shape[1]//2, :]).squeeze(1)

print("✅ Classes defined")


# ============================================================
# CELL 4: Load Data
# ============================================================
print("\nLoading PKLs...")
with open(TRAIN_PKL, "rb") as f:
    tr = pickle.load(f)
X_train = tr["X"];  y_train = tr["y_label"]
del tr; gc.collect()

with open(VAL_PKL, "rb") as f:
    vl = pickle.load(f)
X_val_full = vl["X"];  y_val_full = vl["y_label"]
del vl; gc.collect()

# Val sample 1:6
rng     = np.random.default_rng(RANDOM_SEED)
idx_on  = np.where(y_val_full == 1)[0]
idx_off = np.where(y_val_full == 0)[0]
n_on    = min(N_VAL_ON, len(idx_on))
n_off   = min(n_on * VAL_OFF_MULT, len(idx_off))
s_off   = rng.choice(idx_off, size=n_off, replace=False)
val_idx = np.sort(np.concatenate([idx_on[:n_on], s_off]))
X_val   = X_val_full[val_idx]
y_val   = y_val_full[val_idx]

print(f"✅ Train : {len(X_train):,} windows (50/50)")
print(f"✅ Val   : {len(val_idx):,} | ON={n_on:,} OFF={n_off:,}")

train_ds = NILMDataset(X_train, y_train)
val_ds   = NILMDataset(X_val,   y_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print("✅ DataLoaders ready")


# ============================================================
# CELL 5: Training
# ============================================================
model     = CNNBiLSTM().to(device)
criterion = FocalLoss(alpha=0.75, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1,
)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type=='cuda'))

best_f1, pat_ctr = 0.0, 0
save_path = MODELS_PATH + "washing_machine_v5_final.pt"

print(f"\n{'='*55}")
print(f" Retraining v5 | Saving as washing_machine_v5_final.pt")
print(f"{'='*55}\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            loss = criterion(model(Xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    model.eval()
    probs_l, labs_l = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
                logits = model(Xb.to(device))
            probs_l.append(torch.sigmoid(logits).cpu().numpy())
            labs_l.append(yb.numpy())
    probs  = np.concatenate(probs_l)
    labels = np.concatenate(labs_l)
    preds  = (probs >= 0.50).astype(int)
    f1   = f1_score(labels, preds, pos_label=1, zero_division=0)
    prec = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec  = recall_score(labels, preds, pos_label=1, zero_division=0)

    flag = ""
    if f1 > best_f1:
        best_f1 = f1; pat_ctr = 0
        torch.save(model.state_dict(), save_path)
        flag = " ← ✅ saved"
    else:
        pat_ctr += 1
        flag = f" (patience {pat_ctr}/{PATIENCE})"

    print(f"Ep {epoch:02d}/{EPOCHS} | loss={tr_loss:.4f} | "
          f"F1={f1:.4f} Prec={prec:.4f} Rec={rec:.4f}{flag}")

    if pat_ctr >= PATIENCE:
        print(f"\n⏹ Early stop at epoch {epoch}"); break

print(f"\n✅ Done | Best F1 = {best_f1:.4f}")
print(f"   Model → {save_path}")


# ============================================================
# CELL 6: Full Val Evaluation + Threshold Sweep
# ============================================================
model.load_state_dict(
    torch.load(save_path, map_location=device, weights_only=False))
model.eval()

print("\nFull val inference (623,419 windows)...")
full_ds     = NILMDataset(X_val_full, y_val_full)
full_loader = DataLoader(full_ds, batch_size=1024, shuffle=False,
                         num_workers=2, pin_memory=True)

all_p, all_l = [], []
with torch.no_grad():
    for Xb, yb in full_loader:
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            logits = model(Xb.to(device))
        all_p.append(torch.sigmoid(logits).cpu().numpy())
        all_l.append(yb.numpy())

full_probs  = np.concatenate(all_p)
full_labels = np.concatenate(all_l)

# Prob distribution
print(f"\n  Prob distribution:")
print(f"    min={full_probs.min():.4f}  max={full_probs.max():.4f}")
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    n = int((full_probs >= t).sum())
    print(f"    >= {t}: {n:,} ({100*n/len(full_probs):.2f}%)")

# Threshold sweep
print(f"\n{'Thresh':>7} | {'F1':>6} | {'Prec':>6} | {'Rec':>6} | "
      f"{'TP':>6} | {'FP':>6} | {'FN':>6}")
print("-" * 62)

best_thresh, best_f1_sw = 0.50, 0.0
for thresh in np.arange(0.10, 1.00, 0.05):
    preds = (full_probs >= thresh).astype(int)
    f1    = f1_score(full_labels,  preds, pos_label=1, zero_division=0)
    prec  = precision_score(full_labels, preds, pos_label=1, zero_division=0)
    rec   = recall_score(full_labels,    preds, pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(full_labels, preds, labels=[0,1]).ravel()
    marker = " ←" if f1 > best_f1_sw else ""
    if f1 > best_f1_sw:
        best_f1_sw = f1; best_thresh = thresh
    print(f"  {thresh:.2f}   | {f1:.4f} | {prec:.4f} | {rec:.4f} | "
          f"{tp:>6,} | {fp:>6,} | {fn:>6,}{marker}")

pb  = (full_probs >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(full_labels, pb, labels=[0,1]).ravel()
f1f = f1_score(full_labels, pb, pos_label=1)
prf = precision_score(full_labels, pb, pos_label=1)
rcf = recall_score(full_labels, pb, pos_label=1)

print(f"\n{'='*55}")
print(f" WASHING MACHINE v5 FINAL — Results")
print(f"{'='*55}")
print(f"  Threshold : {best_thresh:.2f}")
print(f"  F1  (ON)  : {f1f:.4f}")
print(f"  Precision : {prf:.4f}")
print(f"  Recall    : {rcf:.4f}")
print(f"  TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"  Model     : {save_path}")
print(f"{'='*55}")

Mounted at /content/drive
✅ Drive mounted
✅ Config | device=cuda
✅ Classes defined

Loading PKLs...
✅ Train : 111,106 windows (50/50)
✅ Val   : 105,483 | ON=15,069 OFF=90,414
✅ DataLoaders ready

 Retraining v5 | Saving as washing_machine_v5_final.pt

Ep 01/40 | loss=0.0615 | F1=0.3466 Prec=0.2107 Rec=0.9768 ← ✅ saved
Ep 02/40 | loss=0.0494 | F1=0.2526 Prec=0.1447 Rec=0.9960 (patience 1/8)
Ep 03/40 | loss=0.0477 | F1=0.2509 Prec=0.1434 Rec=1.0000 (patience 2/8)
Ep 04/40 | loss=0.0473 | F1=0.4118 Prec=0.2617 Rec=0.9649 ← ✅ saved
Ep 05/40 | loss=0.0444 | F1=0.4131 Prec=0.2611 Rec=0.9887 ← ✅ saved
Ep 06/40 | loss=0.0423 | F1=0.6975 Prec=0.6096 Rec=0.8150 ← ✅ saved
Ep 07/40 | loss=0.0367 | F1=0.1165 Prec=0.6909 Rec=0.0636 (patience 1/8)
Ep 08/40 | loss=0.0404 | F1=0.2506 Prec=0.1432 Rec=1.0000 (patience 2/8)
Ep 09/40 | loss=0.0351 | F1=0.6807 Prec=0.5478 Rec=0.8985 (patience 3/8)
Ep 10/40 | loss=0.0284 | F1=0.3508 Prec=0.2144 Rec=0.9644 (patience 4/8)
Ep 11/40 | loss=0.0255 | F1=0.3380 Pre

In [ ]:
# ============================================================
# Washing Machine — Energy Estimation Final
# NILM Project | KASIT - University of Jordan
# Supervised by Dr. Ruba Obiedat
# ============================================================


# ============================================================
# CELL 1: Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")


# ============================================================
# CELL 2: Imports & Config
# ============================================================
import os, gc, pickle, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (f1_score, precision_score,
                             recall_score, confusion_matrix,
                             mean_absolute_error)

BASE_PATH    = "/content/drive/MyDrive/new obada/"
PREPROCESSED = BASE_PATH + "data/preprocessed/"
MODELS_PATH  = BASE_PATH + "models/"

TRAIN_PKL  = PREPROCESSED + "washing_machine_train_v5.pkl"
VAL_PKL    = PREPROCESSED + "washing_machine_val_v5.pkl"
CLF_MODEL  = MODELS_PATH  + "washing_machine_v5_final.pt"
CLF_THRESH = 0.80

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Config | device={device}")
print(f"   Model     : washing_machine_v5_final.pt")
print(f"   Threshold : {CLF_THRESH}")


# ============================================================
# ============================================================
print("\nCalculating watts stats from training data...")

with open(TRAIN_PKL, "rb") as f:
    tr = pickle.load(f)

on_mask  = tr["y_label"] == 1
on_watts = tr["y_watts"][on_mask]
del tr; gc.collect()

AVG_WATTS    = float(np.mean(on_watts))
MEDIAN_WATTS = float(np.median(on_watts))
STD_WATTS    = float(np.std(on_watts))
p25          = float(np.percentile(on_watts, 25))
p75          = float(np.percentile(on_watts, 75))
WATTS_ESTIMATE = AVG_WATTS

print(f"  ON windows : {on_mask.sum():,}")
print(f"  Mean       : {AVG_WATTS:.1f} W  ← ")
print(f"  Median     : {MEDIAN_WATTS:.1f} W")
print(f"  Std        : {STD_WATTS:.1f} W")
print(f"  P25–P75    : {p25:.1f}W – {p75:.1f}W")


# ============================================================
# CELL 4: Load Classification Model
# ============================================================
class CNNBiLSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1,  30, 10, padding=4), nn.BatchNorm1d(30),  nn.ReLU(),
            nn.Conv1d(30, 30,  8, padding=3), nn.BatchNorm1d(30),  nn.ReLU(),
            nn.Conv1d(30, 40,  6, padding=2), nn.BatchNorm1d(40),  nn.ReLU(),
            nn.Conv1d(40, 50,  5, padding=2), nn.BatchNorm1d(50),  nn.ReLU(),
            nn.Conv1d(50, 50,  5, padding=2), nn.BatchNorm1d(50),  nn.ReLU(),
        )
        self.lstm = nn.LSTM(50, 128, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )
    def forward(self, x):
        f = self.cnn(x).permute(0, 2, 1)
        o, _ = self.lstm(f)
        return self.fc(o[:, o.shape[1]//2, :]).squeeze(1)

model = CNNBiLSTMClassifier().to(device)
missing, unexpected = model.load_state_dict(
    torch.load(CLF_MODEL, map_location=device, weights_only=False),
    strict=True)
model.eval()
print(f"\n✅ Model loaded")
print(f"   Missing={missing} | Unexpected={unexpected}")


# ============================================================
# CELL 5: Full Val Inference
# ============================================================
print("\nLoading val PKL...")
with open(VAL_PKL, "rb") as f:
    vl = pickle.load(f)
X_val_full  = vl["X"]
y_val_watts = vl["y_watts"]
y_val_label = vl["y_label"]
del vl; gc.collect()

class SimpleDataset(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i]

full_loader = DataLoader(SimpleDataset(X_val_full),
                         batch_size=1024, shuffle=False,
                         num_workers=2, pin_memory=True)

print("Running inference on 623,419 windows...")
all_probs = []
with torch.no_grad():
    for Xb in full_loader:
        with torch.amp.autocast('cuda', enabled=(device.type=='cuda')):
            logits = model(Xb.to(device))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())

clf_probs  = np.concatenate(all_probs)
clf_labels = (clf_probs >= CLF_THRESH).astype(int)

print(f"\n  Prob distribution:")
print(f"    min={clf_probs.min():.4f}  max={clf_probs.max():.4f}")
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    n = int((clf_probs >= t).sum())
    print(f"    >= {t}: {n:,} ({100*n/len(clf_probs):.2f}%)")


# ============================================================
# CELL 6: Energy Estimation + Full Report
# ============================================================
# Two-stage: ON → WATTS_ESTIMATE, OFF → 0W
estimated_watts = np.where(clf_labels == 1, WATTS_ESTIMATE, 0.0)

# ── Classification metrics ─────────────────────────────────
f1   = f1_score(y_val_label,  clf_labels, pos_label=1, zero_division=0)
prec = precision_score(y_val_label, clf_labels, pos_label=1, zero_division=0)
rec  = recall_score(y_val_label,    clf_labels, pos_label=1, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_val_label, clf_labels, labels=[0,1]).ravel()

# MAE on ON windows ─────────────────────────────────
on_true_mask = y_val_label == 1
true_on_watts = y_val_watts[on_true_mask]
pred_on_watts = estimated_watts[on_true_mask]
mae_on = mean_absolute_error(true_on_watts, pred_on_watts)

# MAE on detected ON windows only
detected_mask = clf_labels == 1
if detected_mask.sum() > 0:
    mae_detected = mean_absolute_error(
        y_val_watts[detected_mask],
        estimated_watts[detected_mask]
    )
else:
    mae_detected = float('nan')

# ── Energy metrics ─────────────────────────────────────────
true_kwh = float(y_val_watts.sum() / 60 / 1000)
pred_kwh = float(estimated_watts.sum() / 60 / 1000)
kwh_err  = abs(true_kwh - pred_kwh)
kwh_pct  = 100 * kwh_err / max(true_kwh, 1e-8)

# ON minutes
true_on_minutes = int(y_val_label.sum())
pred_on_minutes = int(clf_labels.sum())

print(f"\n{'='*55}")
print(f" WASHING MACHINE — Final Energy Report")
print(f"{'='*55}")
print(f"\n  Stage 1 — Classification:")
print(f"    Threshold  = {CLF_THRESH}")
print(f"    F1         = {f1:.4f}")
print(f"    Precision  = {prec:.4f}")
print(f"    Recall     = {rec:.4f}")
print(f"    TP={tp:,}  FP={fp:,}  FN={fn:,}  TN={tn:,}")
print(f"\n  Stage 2 — Energy Estimation:")
print(f"    Method     = Mean watts from training")
print(f"    Estimate   = {WATTS_ESTIMATE:.1f} W per ON minute")
print(f"    MAE (all ON windows)      = {mae_on:.1f} W")
print(f"    MAE (detected ON windows) = {mae_detected:.1f} W")
print(f"\n  ON Minutes:")
print(f"    True  ON = {true_on_minutes:,} minutes")
print(f"    Pred  ON = {pred_on_minutes:,} minutes")
print(f"    Error    = {abs(true_on_minutes-pred_on_minutes):,} minutes")
print(f"\n  Energy (full val period):")
print(f"    True  kWh = {true_kwh:.3f} kWh")
print(f"    Pred  kWh = {pred_kwh:.3f} kWh")
print(f"    Error     = {kwh_err:.3f} kWh ({kwh_pct:.1f}%)")
print(f"{'='*55}")

# ── Example for user ───────────────────────────────────────
example_min = 60
example_kwh = example_min * WATTS_ESTIMATE / 60 / 1000
print(f"\n  Example (60-minute wash cycle):")
print(f"    'Washing machine ran for 60 minutes'")
print(f"    'Consumed approximately {example_kwh:.3f} kWh'")
print(f"    'MAE ≈ ±{mae_on:.0f}W per minute'")

# ── Save stats ─────────────────────────────────────────────
stats = {
    "appliance"          : "washing_machine",
    "clf_model"          : "washing_machine_v5_final.pt",
    "clf_threshold"      : CLF_THRESH,
    "watts_estimate"     : round(WATTS_ESTIMATE, 1),
    "watts_median"       : round(MEDIAN_WATTS, 1),
    "watts_std"          : round(STD_WATTS, 1),
    "estimation_method"  : "mean_from_training",
    "val_f1"             : round(f1, 4),
    "val_precision"      : round(prec, 4),
    "val_recall"         : round(rec, 4),
    "val_mae_on_watts"   : round(mae_on, 1),
    "val_kwh_error_pct"  : round(kwh_pct, 1),
}

stats_path = MODELS_PATH + "washing_machine_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)

print(f"\n✅ Stats saved → {stats_path}")
print(f"\n{'='*55}")
print(f" PROJECT SUMMARY — All Appliances")
print(f"{'='*55}")
print(f"  {'Appliance':<20} {'F1':>6} {'MAE':>10}")
print(f"  {'-'*40}")
print(f"  {'Kettle':<20} {'0.63':>6} {'N/A':>10}")
print(f"  {'Fridge':<20} {'0.53':>6} {'N/A':>10}")
print(f"  {'Microwave':<20} {'0.24':>6} {'N/A':>10}")
print(f"  {'Dishwasher':<20} {'0.73':>6} {'N/A':>10}")
print(f"  {'Washing Machine':<20} {f1:>6.2f} {mae_on:>8.1f}W")
print(f"{'='*55}")


Mounted at /content/drive
✅ Drive mounted
✅ Config | device=cuda
   Model     : washing_machine_v5_final.pt
   Threshold : 0.8

Calculating watts stats from training data...
  ON windows : 55,553
  Mean       : 434.3 W  ← نستخدمه
  Median     : 110.0 W
  Std        : 699.6 W
  P25–P75    : 66.3W – 280.0W

✅ Model loaded
   Missing=[] | Unexpected=[]

Loading val PKL...
Running inference on 623,419 windows...

  Prob distribution:
    min=0.0001  max=0.9878
    >= 0.5: 32,768 (5.26%)
    >= 0.6: 26,930 (4.32%)
    >= 0.7: 21,752 (3.49%)
    >= 0.8: 15,935 (2.56%)
    >= 0.9: 7,372 (1.18%)

 WASHING MACHINE — Final Energy Report

  Stage 1 — Classification:
    Threshold  = 0.8
    F1         = 0.5767
    Precision  = 0.5610
    Recall     = 0.5933
    TP=8,940  FP=6,995  FN=6,129  TN=601,355

  Stage 2 — Energy Estimation:
    Method     = Mean watts from training
    Estimate   = 434.3 W per ON minute
    MAE (all ON windows)      = 413.3 W
    MAE (detected ON windows) = 485.7 W

  ON